# Bài 13 · Trực quan hoá nâng cao và phản biện biểu đồ

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Viện TTNT, UET-VNU**

> 💡 File → **Save a copy in Drive** trước khi sửa.

**Mục tiêu bài học.** Sau notebook này, bạn sẽ:

1. Vẽ hình thống kê nhiều chiều bằng **seaborn** (boxplot, histplot-hue, heatmap).
2. Tạo **bản đồ choropleth** từ GeoJSON của Inside Airbnb và nhận biết bẫy diện tích.
3. Làm quen với plotly express để tạo biểu đồ tương tác.
4. **Phản biện và sửa** ba lỗi trình bày thường gặp: trục kép, biểu đồ tròn nhiều lát và hai đơn vị trên một trục.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", rc={"grid.alpha": 0.3})
XANH, CAM = "#1E93AB", "#E8890C"

BASE = "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29"
df = pd.read_csv(f"{BASE}/visualisations/listings.csv")
df = df[df["price"].notna() & (df["price"] > 0)]
print(df.shape)

## 1. seaborn — so sánh phân phối giữa các nhóm

In [ ]:
thu_tu = ["Shared room", "Private room", "Entire home/apt", "Hotel room"]
fig, ax = plt.subplots(figsize=(8.6, 3.8))
sns.boxplot(data=df, x="price", y="room_type", order=thu_tu,
            color=XANH, showfliers=False, width=0.55, ax=ax)
ax.set_xscale("log")
ax.set_xlabel("giá (CLP/đêm, thang log)"); ax.set_ylabel("")
ax.set_title("Các loại phòng có mức giá khác nhau nhưng phân phối chồng lấn đáng kể",
             loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

Boxplot cho thấy **độ phân tán** và **độ chồng lấn** giữa các nhóm, những thông tin không có trong bảng trung vị.
`showfliers=False` ẩn các điểm ngoại lai (outlier) để tập trung vào phần chính của mỗi phân phối.

In [ ]:
# hue tách nhóm bằng màu để so sánh hình dạng phân phối giá của hai quận
hai_quan = df[df["neighbourhood"].isin(["Santiago", "Las Condes"])]
fig, ax = plt.subplots(figsize=(8.6, 3.6))
sns.histplot(data=hai_quan, x="price", hue="neighbourhood", log_scale=True,
             element="step", stat="density", common_norm=False,
             palette=[XANH, CAM], ax=ax)
ax.set_title("Las Condes: phân phối dịch phải rõ rệt (trung vị gấp khoảng 2 lần)",
             loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# heatmap thể hiện bảng pivot bằng màu
quan_lon = df["neighbourhood"].value_counts().head(8).index
pv = (df[df["neighbourhood"].isin(quan_lon)]
      .pivot_table(values="price", index="neighbourhood", columns="room_type",
                   aggfunc="median")[["Entire home/apt", "Private room"]] / 1000)

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(pv.sort_values("Entire home/apt", ascending=False),
            annot=True, fmt=".0f", cmap="Blues",
            cbar_kws={"label": "giá trung vị (nghìn CLP)"}, ax=ax)
ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); plt.show()

## 2. Bản đồ choropleth

In [ ]:
import geopandas as gpd

geo = gpd.read_file(f"{BASE}/visualisations/neighbourhoods.geojson")
kpi = df.groupby("neighbourhood").agg(gia=("price", "median"), n=("price", "size")).reset_index()
ban_do = geo.merge(kpi, on="neighbourhood", how="left")

fig, ax = plt.subplots(figsize=(8, 5.4))
ban_do.plot(column="gia", cmap="Blues", legend=True, edgecolor="#999", linewidth=0.5, ax=ax,
            legend_kwds={"label": "giá trung vị (CLP/đêm)", "shrink": 0.7},
            missing_kwds={"color": "#eee"})
ax.set_axis_off()
ax.set_title("Giá trung vị cao hơn ở khu vực đông bắc Santiago", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

⚠️ **Bẫy diện tích:** Lo Barnechea có diện tích lớn và màu đậm nhất nhưng chỉ có khoảng 800 chỗ ở;
quận Santiago nhỏ hơn nhiều nhưng có gần 7.000 chỗ ở. Vì diện tích lớn dễ thu hút sự chú ý,
bản đồ choropleth cần đi kèm cỡ mẫu `n` của từng quận:

In [ ]:
kpi.nlargest(5, "n")[["neighbourhood", "n", "gia"]]

## 3. plotly express — xem chi tiết khi rê chuột

In [ ]:
import plotly.express as px

s = df.sample(3000, random_state=1)
fig = px.scatter(s, x="longitude", y="latitude", color="room_type",
                 hover_name="name", hover_data={"price": ":,.0f", "neighbourhood": True},
                 opacity=0.55, title="Rê chuột để xem chi tiết của 3.000 chỗ ở mẫu")
fig.update_layout(height=420)
fig.show()

Biểu đồ tương tác phù hợp để **khám phá** dữ liệu. Báo cáo PDF của bài tập lớn vẫn dùng hình tĩnh
và chú thích thay cho thao tác rê chuột. Dashboard HTML (`fig.write_html("dashboard.html")`) có thể dùng làm sản phẩm điểm thưởng.

## 4. Phản biện ba biểu đồ lỗi

Ba ô mã dưới tạo ra ba biểu đồ có lỗi trình bày thường gặp. Ở mục 5, bạn sẽ xác định lỗi
và sửa từng biểu đồ.

> Các chuỗi **giá** ở hình A và C là số minh hoạ tự sinh — không phải giá Airbnb thật; chỉ **số đánh giá** ở hình A mới là dữ liệu thật.

In [ ]:
# Hình lỗi A: trục kép tạo cảm giác có tương quan
rv = pd.read_csv(f"{BASE}/visualisations/reviews.csv", parse_dates=["date"])
thang = rv[rv["date"] < "2026-07-01"].set_index("date").resample("ME").size().loc["2024":]
gia_gia_lap = pd.Series(np.linspace(55, 62, len(thang))
                        + np.random.default_rng(3).normal(0, 1.2, len(thang)), index=thang.index)

fig, ax1 = plt.subplots(figsize=(8.6, 3.6))
ax1.plot(thang.index, thang.values, color=XANH, lw=2)
ax1.set_ylabel("số đánh giá", color=XANH)
ax2 = ax1.twinx()
ax2.plot(gia_gia_lap.index, gia_gia_lap.values, color="red", ls="--", lw=2)
ax2.set_ylabel("giá trung vị (nghìn CLP)", color="red")
ax2.set_ylim(54, 63)
ax1.set_title("Phân tích nhu cầu và giá", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Hình lỗi B: biểu đồ tròn gồm 12 lát và nhiều màu
quan12 = df["neighbourhood"].value_counts().head(12)
fig, ax = plt.subplots(figsize=(6.6, 4.4))
ax.pie(quan12.values, labels=quan12.index, autopct="%1.1f%%",
       colors=plt.cm.tab20.colors, startangle=90, textprops={"fontsize": 8})
ax.set_title("Phân bố chỗ ở theo quận", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Hình lỗi C: hai tiền tệ, một trục
idx = pd.date_range("2024-07-31", periods=24, freq="ME")
scl = (np.linspace(52, 65, 24) + np.random.default_rng(1).normal(0, 1.5, 24)) * 1000  # CLP
rio = np.linspace(280, 340, 24) + np.random.default_rng(2).normal(0, 8, 24)           # BRL

fig, ax = plt.subplots(figsize=(8.6, 3.4))
ax.plot(idx, scl, lw=2, color=XANH, label="Santiago (CLP)")
ax.plot(idx, rio, lw=2, color=CAM, label="Rio (BRL)")
ax.set_title("So sánh giá: Santiago và Rio", fontweight="bold")
ax.legend(); plt.tight_layout(); plt.show()

## 5. Bài tập tại lớp — hồ sơ lỗi và bản sửa

### Bài 1 — Lập hồ sơ lỗi

Với **mỗi** hình lỗi, hãy ghi vào ô Markdown bên dưới: (a) kết luận mà hình gợi ra,
(b) lựa chọn trình bày nào tạo ra ấn tượng đó và (c) một câu hỏi để kiểm chứng.
Dùng năm tiêu chí: *trục, đơn vị, cỡ mẫu n, dạng biểu đồ và lời diễn giải*.

*(Viết hồ sơ lỗi của bạn vào đây; nhấp đúp vào ô để sửa.)*

- **Hình lỗi A:** …
- **Hình lỗi B:** …
- **Hình lỗi C:** …

### Bài 2 — Sửa hình lỗi A

Tách biểu đồ trục kép thành **hai khung nhỏ chung trục thời gian** (mỗi trục *y* một thang riêng).
Vì sao cách này trung thực hơn: không thể "kéo" hai đường trông đồng biến chỉ bằng cách chỉnh `ylim`.

In [ ]:
# TODO Bài 2:
fig, (axA, axB) = plt.subplots(2, 1, sharex=True, figsize=(8.6, 4))
axA.plot(thang.index, thang.values, color=XANH, lw=2)
axA.set_ylabel("số đánh giá")
axB.plot(gia_gia_lap.index, gia_gia_lap.values, color=CAM, lw=2)
axB.set_ylabel("giá trung vị\n(nghìn CLP · minh hoạ)")
axA.set_title("Bản sửa: tách hai khung, chung trục thời gian", loc="left", fontweight="bold")
for a in (axA, axB):
    a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

### Bài 3 — Sửa hình lỗi B

Vẽ lại biểu đồ tròn 12 lát thành **biểu đồ thanh ngang xếp hạng** như ở bài 12.
Chỉ nhấn một thanh, ghi nhãn tỷ lệ và đặt tiêu đề nêu thông điệp.

In [ ]:
# TODO Bài 3:
ty_le = (quan12 / len(df) * 100).sort_values()
fig, ax = plt.subplots(figsize=(8, 4.2))
mau = [CAM if q == ty_le.idxmax() else XANH for q in ty_le.index]
bars = ax.barh(ty_le.index, ty_le.values, color=mau, height=0.6)
ax.bar_label(bars, [f" {v:.1f}%" for v in ty_le.values], fontsize=9)
ax.set_title("Gần 2 phần 5 số chỗ ở nằm ở quận trung tâm Santiago", loc="left", fontweight="bold")
ax.set_xlabel("% tổng chỗ ở (n = {:,})".format(len(df)))
ax.grid(axis="y", alpha=0)
plt.tight_layout(); plt.show()

### Bài 4 — Sửa hình lỗi C

Quy hai chuỗi giá về **chỉ số với tháng đầu = 100** rồi vẽ trên cùng một trục. Khi đó,
hai chuỗi có cùng đơn vị và có thể dùng để so nhịp thay đổi. Vì sao vẫn không được kết luận
thành phố nào tăng nhanh hơn trong thực tế khi cả hai chuỗi đều là số liệu minh hoạ?

In [ ]:
# TODO Bài 4:
fig, ax = plt.subplots(figsize=(8.6, 3.4))
ax.plot(idx, scl / scl[0] * 100, lw=2.2, color=XANH, label="Santiago")
ax.plot(idx, rio / rio[0] * 100, lw=2.2, color=CAM, label="Rio")
ax.axhline(100, color="#777", lw=1, ls="--")
ax.set_ylabel("chỉ số giá (tháng đầu = 100)")
ax.set_title("Cùng quy về mốc 100 mới so được nhịp tăng (số liệu minh hoạ)", loc="left", fontweight="bold")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 6. Bài tập về nhà — Choropleth cho thành phố của nhóm

1. Tải `neighbourhoods.geojson` + `visualisations/listings.csv` của thành phố nhóm bạn nhận cho bài tập lớn.
2. Vẽ hai bản đồ cạnh nhau: (a) giá trung vị theo quận và (b) **số chỗ ở** theo quận.
   Hai bản đồ bổ sung thông tin về giá trị và quy mô nguồn cung.
3. Viết năm câu trình bày khu nào đắt, khu nào có nhiều chỗ ở, hai bản đồ cung cấp thông tin
   khác nhau thế nào và bẫy diện tích ảnh hưởng ra sao.

In [ ]:
RUN_CHALLENGE = False
if RUN_CHALLENGE:
    CITY_BASE = "https://data.insideairbnb.com/..."
    ...

---

## Tóm tắt bài học

| Nội dung chính | Vì sao quan trọng |
|---|---|
| seaborn nhận DataFrame và tên cột | So sánh phân phối giữa các nhóm trong một hình |
| Choropleth cần dữ liệu vùng, chỉ số đã gộp và cỡ mẫu n | Tránh diễn giải sai do bẫy diện tích |
| Biểu đồ tương tác để khám phá; biểu đồ tĩnh để xuất bản | Chọn dạng phù hợp với mục đích sử dụng |
| Năm tiêu chí: trục, đơn vị, n, dạng biểu đồ và lời diễn giải | Phản biện cả hình tự tạo và hình do AI sinh |

**Bài sau:** xây dựng **câu chuyện dữ liệu** và thẩm định một bản phân tích hoàn chỉnh do AI viết.